<a href="https://colab.research.google.com/github/isak-sjoholm/running_dashboard/blob/main/Running_Dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

INSTALL LIBRARIES

In [1]:
import os
#os.environ["STRAVA_CLIENT_ID"] = ""
#os.environ["STRAVA_CLIENT_SECRET"] = ""
#os.environ["STRAVA_REFRESH_TOKEN"] = ""

#from google.colab import userdata


In [2]:
# @title
## INSTALL LIBRARIES
!pip install requests
!pip install pandas
!pip install polyline
!pip install python-dotenv

IMPORT LIBRARIES


In [3]:
# @title
## IMPORT LIBRARIES
import urllib.parse
import requests
import json
import time
import pandas as pd
from datetime import datetime, timezone
import plotly.graph_objects as go
import plotly.express as px
import polyline
from plotly.subplots import make_subplots
import plotly.io as pio
import os



DEFINIERA ALLA FUNKTIONER

In [4]:
# Hjälpfunktioner
def seconds_to_pace(seconds):
    minutes, secs = divmod(int(seconds), 60)
    return f"{minutes}:{secs:02d}"

def get_seconds(pace_str):
    if pace_str == 'N/A':
        return None
    parts = pace_str.split(':')
    if len(parts) == 2:
        return int(parts[0]) * 60 + int(parts[1])
    else:
        return int(parts[0]) * 3600 + int(parts[1]) * 60 + int(parts[2])

def seconds_to_time(seconds):
    hours, remainder = divmod(int(seconds), 3600)
    minutes, secs = divmod(remainder, 60)
    if hours > 0:
        return f"{hours}:{minutes:02d}:{secs:02d}"
    return f"{minutes}:{secs:02d}"

def get_row_colors(row_paces):
    seconds_list = [get_seconds(p) for p in row_paces]
    valid = [s for s in seconds_list if s is not None]
    if not valid:
        return ['lightgray'] * len(row_paces)
    best = min(valid)
    worst = max(valid)
    colors = []
    for s in seconds_list:
        if s is None:
            colors.append('lightgray')
        elif best == worst:
            colors.append('rgb(0,200,0)')
        else:
            ratio = (s - best) / (worst - best)
            if ratio < 0.5:
                r = int(255 * ratio * 2)
                g = 200
            else:
                r = 255
                g = int(200 * (1 - (ratio - 0.5) * 2))
            colors.append(f'rgb({r},{g},0)')
    return colors

DEFINIERA MÅLSÄTTNINGAR

In [5]:
weekly_mileage_goal = 30
weekly_runs_goal = 4
HR_Max = 195

KOPPLA UPP STRAVA API


In [6]:
# @title
# KOPPLA UPP STRAVA API

CLIENT_ID = os.environ["STRAVA_CLIENT_ID"]
CLIENT_SECRET = os.environ["STRAVA_CLIENT_SECRET"]
REFRESH_TOKEN = os.environ["STRAVA_REFRESH_TOKEN"]


def refresh_access_token(client_id: str, client_secret: str, refresh_token: str) -> dict:
    response = requests.post(
        "https://www.strava.com/oauth/token",
        data={
            "client_id": client_id,
            "client_secret": client_secret,
            "grant_type": "refresh_token",
            "refresh_token": refresh_token,
        },
        timeout=30,
    )
    response.raise_for_status()
    return response.json()

token_data = refresh_access_token(CLIENT_ID, CLIENT_SECRET, REFRESH_TOKEN)
access_token = token_data["access_token"]
new_refresh_token = token_data["refresh_token"]

FETCH ALL STRAVA ACTIVITIES


In [7]:
# @title
# FETCH ALL STRAVA ACTIVITIES

def get_all_activities(access_token: str, per_page: int = 200) -> list[dict]:
    headers = {"Authorization": f"Bearer {access_token}"}
    activities = []
    page = 1

    while True:
        response = requests.get(
            "https://www.strava.com/api/v3/athlete/activities",
            headers=headers,
            params={"per_page": per_page, "page": page},
            timeout=30,
        )
        response.raise_for_status()
        batch = response.json()

        if not batch:
            break

        activities.extend(batch)

        if len(batch) < per_page:
            break

        page += 1
        time.sleep(0.25)

    return activities

activities = get_all_activities(access_token)
df_activities = pd.DataFrame(activities)


FÖRBERED STRAVA-DATA FÖR ANALYS

In [8]:
# @title
# Filtrera bort allt som inte är löpning
df = df_activities[df_activities['sport_type'] == 'Run'].copy()

# Sätt datum-data till korrekt format
df['start_date_local'] = pd.to_datetime(df['start_date_local'])

# Räkan om distans till km
df['distance_km'] = df['distance'] / 1000

# Sätt datum-data till korrekt format
df['year'] = df['start_date_local'].dt.isocalendar().year
df['week'] = df['start_date_local'].dt.isocalendar().week
df['year_week'] = df['start_date_local'].dt.to_period('W')

# Sätt dagens datum, år & vecka
today = pd.Timestamp.now(tz='UTC')
current_year = today.year
current_week = today.isocalendar().week

df['summary_polyline'] = df['map'].apply(lambda x: x.get('summary_polyline') if isinstance(x, dict) else None)


/tmp/ipykernel_1379/2920793058.py:14: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df['year_week'] = df['start_date_local'].dt.to_period('W')


**`KPI 1: KM PER WEEK`**

In [9]:
# @title
# Aggregera km per vecka
weekly_km = df.groupby(['year', 'week'])['distance_km'].sum().reset_index()
weekly_km = weekly_km.sort_values(['year', 'week'])

# Ta fram nuvarande vecka
this_week = weekly_km[(weekly_km['year'] == current_year) & (weekly_km['week'] == current_week)]
km_this_week = this_week['distance_km'].values[0] if len(this_week) > 0 else 0

# Ta fram snitt senaste 10 veckorna (ex. denna vecka)
last_10 = weekly_km[~((weekly_km['year'] == current_year) & (weekly_km['week'] == current_week))].tail(10)
avg_10_weeks = last_10['distance_km'].mean()


# Skapa komplett veckoindex för senaste 52 veckor
all_weeks = pd.date_range(end=today, periods=52, freq='W-MON')
all_weeks_df = pd.DataFrame({
    'year': all_weeks.isocalendar().year.values,
    'week': all_weeks.isocalendar().week.values
})

# Merga med faktisk data, fyll 0 för tomma veckor
weekly_km_full = all_weeks_df.merge(weekly_km, on=['year', 'week'], how='left').fillna(0)


In [10]:
# @title
## Ta fram mileage senaste 52 veckor
last_52 = weekly_km_full.tail(52).copy()

## Labela senaste 52 veckor med veckonummer
last_52['label'] = last_52['year'].astype(str) + '-W' + last_52['week'].astype(str).str.zfill(2)

## Räkan rolling average för mileage
last_52['rolling_avg'] = last_52['distance_km'].rolling(window=10, min_periods=1).mean()

# Plotta en bar chart
fig_km = go.Figure()

fig_km.add_trace(go.Bar(x=last_52['label'], y=last_52['distance_km'], name='Km per vecka'))

# Lägg till en linje för målsättning
fig_km.add_trace(go.Scatter( x=last_52['label'], y=[weekly_mileage_goal] * len(last_52), mode='lines', name=f'Målsättning', line=dict(color='green', dash='dash')))


# Lägg till 10-veckors-snittet som en linje
fig_km.add_trace(go.Scatter(x=last_52['label'], y=last_52['rolling_avg'], mode='lines', name='Rullande snitt 10v', line=dict(color='red')))

# Lägg till titlar osv.
fig_km.update_layout(
    title=f'Km per vecka | Denna vecka: {km_this_week:.1f} km | Snitt last 10w: {avg_10_weeks:.1f} km',
    xaxis_title='Vecka',
    yaxis_title='Km',
    xaxis_tickangle=-45
)

fig_km.show()


**`KPI 2: RUNS PER WEEK`**

In [11]:
# @title
# Aggregera runs per vecka
weekly_runs = df.groupby(['year', 'week']).size().reset_index(name='run_count')
weekly_runs = weekly_runs.sort_values(['year', 'week'])
weekly_runs_full = all_weeks_df.merge(weekly_runs, on=['year', 'week'], how='left').fillna(0)

# Ta fram nuvarande vecka
this_week_runs = weekly_runs[(weekly_runs['year'] == current_year) & (weekly_runs['week'] == current_week)]
runs_this_week = this_week_runs['run_count'].values[0] if len(this_week_runs) > 0 else 0

# Ta fram nitt senaste 10 veckorna (ex. denna vecka)
last_10 = weekly_runs[~((weekly_runs['year'] == current_year) & (weekly_runs['week'] == current_week))].tail(10)
avg_10_weeks_runs = last_10['run_count'].mean()



In [12]:
# @title
## Ta fram mileage senaste 52 veckor
last_52 = weekly_runs_full.copy()

## Labela senaste 52 veckor med veckonummer
last_52['label'] = last_52['year'].astype(str) + '-W' + last_52['week'].astype(str).str.zfill(2)

## Räkan rolling average för mileage
last_52['rolling_avg'] = last_52['run_count'].rolling(window=10, min_periods=1).mean()

# Plotta en bar chart
fig_runs = go.Figure()

fig_runs.add_trace(go.Bar(x=last_52['label'], y=last_52['run_count'], name='Pass per vecka', marker_color='orange'))

# Lägg till en linje för målsättning
fig_runs.add_trace(go.Scatter(x=last_52['label'], y=[weekly_runs_goal] * len(last_52), mode='lines', name='Målsättning', line=dict(color='darkgreen', dash='dash')))

# Lägg till 10-veckors-snittet som en linje
fig_runs.add_trace(go.Scatter(x=last_52['label'], y=last_52['rolling_avg'], mode='lines', name='Rullande snitt 10v', line=dict(color='darkred')))

# Lägg till titlar osv.
fig_runs.update_layout(
    title=f'Pass per vecka | Denna vecka: {runs_this_week:.0f} pass | Snitt last 10w: {avg_10_weeks_runs:.1f} pass',
    xaxis_title='Vecka',
    yaxis_title='Antal pass',
    xaxis_tickangle=-45
)

fig_runs.show()


**`KPI 2: RUNS PER WEEK`**


Hämta detaljdata för PR-pass

In [13]:
# @title
def get_best_efforts(activity_id, access_token):
    response = requests.get(
        f"https://www.strava.com/api/v3/activities/{activity_id}",
        headers={"Authorization": f"Bearer {access_token}"},
        timeout=30
    )
    response.raise_for_status()
    data = response.json()
    return data.get('best_efforts', [])

# Runs med PR
pr_activities = df[df['pr_count'] > 0]['id'].tolist()


all_best_efforts = []

# Hämta detaljdata för aktiviteter med PR
for activity_id in pr_activities:
    efforts = get_best_efforts(activity_id, access_token)
    for effort in efforts:
        effort['activity_id'] = activity_id
        all_best_efforts.append(effort)
    time.sleep(0.25)

df_efforts = pd.DataFrame(all_best_efforts)


Ta fram PR på alla distanser

In [14]:
# @title


# Filtrera bara PR-runs
df_pr = df_efforts[df_efforts['pr_rank'] == 1].copy()
df_pr['start_date_local'] = pd.to_datetime(df_pr['start_date_local'])

# Definiera distanser
distances = ['5K', '10K', 'Half-Marathon', 'Marathon']

# Ta fram PR per distans
pr_results = {}

for dist in distances:
    pr = df_pr[df_pr['name'] == dist].sort_values('elapsed_time').head(1)
    if len(pr) > 0:
        seconds = int(pr['elapsed_time'].values[0])
        minutes, secs = divmod(seconds, 60)
        hours, minutes = divmod(minutes, 60)
        date = pr['start_date_local'].dt.date.values[0]
        time_str = f"{hours}:{minutes:02d}:{secs:02d}" if hours > 0 else f"{minutes}:{secs:02d}"
        pr_results[dist] = {'time': time_str, 'date': date, 'seconds': seconds}
        print(f"{dist}: {time_str} | {date}")
    else:
        pr_results[dist] = {'time': 'N/A', 'date': 'N/A', 'seconds': None}
        print(f"{dist}: Ingen data")


5K: 24:46 | 2024-05-07
10K: 53:31 | 2024-05-05
Half-Marathon: 2:03:05 | 2026-04-25
Marathon: Ingen data


Ta fram bästa tid senaste 6 veckor på alla distanser

In [15]:
# @title
# Best last 6 weeks per distans
six_weeks_ago = today - pd.Timedelta(weeks=6)

best_6w = {}
for dist in distances:
    recent = df_efforts[
        (df_efforts['name'] == dist) &
        (pd.to_datetime(df_efforts['start_date_local']) >= six_weeks_ago)
    ].sort_values('elapsed_time').head(1)

    if len(recent) > 0:
        seconds = int(recent['elapsed_time'].values[0])
        minutes, secs = divmod(seconds, 60)
        hours, minutes = divmod(minutes, 60)
        time_str = f"{hours}:{minutes:02d}:{secs:02d}" if hours > 0 else f"{minutes}:{secs:02d}"
        best_6w[dist] = {'time': time_str, 'seconds': seconds}
        print(f"{dist} senaste 6v: {time_str}")
    else:
        best_6w[dist] = {'time': 'N/A', 'seconds': None}
        print(f"{dist} senaste 6v: N/A")

5K senaste 6v: 26:05
10K senaste 6v: 56:13
Half-Marathon senaste 6v: 2:03:05
Marathon senaste 6v: N/A


Tabell över PRs & Form

In [16]:
# @title
table_distances = ['5K', '10K', 'Half-Marathon', 'Marathon']

# Beräkna färg baserat på hur nära PR
def get_color(pr_seconds, recent_seconds):
    if pr_seconds is None or recent_seconds is None:
        return 'lightgray'
    diff = recent_seconds - pr_seconds
    max_diff = pr_seconds * 0.2  # 20% sämre än PR = mörkast röd
    ratio = min(diff / max_diff, 1.0)
    r = int(255 * ratio)
    g = int(255 * (1 - ratio))
    return f'rgb({r},{g},0)'


# Nike pace-tabell
pace_table = [
    {'5K': '17:05', '10K': '35:45', 'Half-Marathon': '1:18:00', 'Marathon': '2:44:00'},
    {'5K': '18:45', '10K': '39:00', 'Half-Marathon': '1:25:00', 'Marathon': '3:00:00'},
    {'5K': '20:15', '10K': '42:00', 'Half-Marathon': '1:35:00', 'Marathon': '3:15:00'},
    {'5K': '22:00', '10K': '45:45', 'Half-Marathon': '1:40:00', 'Marathon': '3:30:00'},
    {'5K': '23:45', '10K': '49:00', 'Half-Marathon': '1:50:00', 'Marathon': '3:45:00'},
    {'5K': '25:15', '10K': '52:30', 'Half-Marathon': '1:55:00', 'Marathon': '4:00:00'},
    {'5K': '27:00', '10K': '55:50', 'Half-Marathon': '2:05:00', 'Marathon': '4:15:00'},
    {'5K': '28:30', '10K': '59:00', 'Half-Marathon': '2:10:00', 'Marathon': '4:30:00'},
    {'5K': '30:00', '10K': '62:30', 'Half-Marathon': '2:20:00', 'Marathon': '4:45:00'},
    {'5K': '31:45', '10K': '66:00', 'Half-Marathon': '2:25:00', 'Marathon': '5:00:00'},
    {'5K': '33:00', '10K': '69:00', 'Half-Marathon': '2:35:00', 'Marathon': '5:15:00'},
    {'5K': '35:00', '10K': '72:00', 'Half-Marathon': '2:40:00', 'Marathon': '5:30:00'},
    {'5K': '36:15', '10K': '75:00', 'Half-Marathon': '2:50:00', 'Marathon': '5:40:00'},
    {'5K': '38:00', '10K': '78:30', 'Half-Marathon': '2:55:00', 'Marathon': '5:50:00'},
    {'5K': '39:30', '10K': '81:30', 'Half-Marathon': '3:05:00', 'Marathon': '6:00:00'},
]

# Omvandla paces från tabellen till sekunder
pace_table_seconds = [{k: get_seconds(v) for k, v in row.items()} for row in pace_table]

def get_estimated_time(known_dist, known_seconds, target_dist):
    if known_seconds is None:
        return 'N/A'
    # Hitta rätt rad — avrunda ner till närmsta rad
    best_row = None
    for row in pace_table_seconds:
        if row[known_dist] <= known_seconds:
            best_row = row
        else:
            break
    if best_row is None:
        best_row = pace_table_seconds[0]
    est_seconds = best_row[target_dist]
    return seconds_to_time(est_seconds)


# Ta fram estimated race times
# Distansordning för fallback (längsta neråt)
dist_order = ['Marathon', 'Half-Marathon', '10K', '5K']

estimated_times = []
for target in table_distances:
    # Steg 1: Har distansen sprungits senaste 6v?
    if best_6w[target]['seconds'] is not None:
        estimated_times.append(best_6w[target]['time'])
        continue

    # Steg 2: Hitta närmsta distans neråt som sprungits senaste 6v
    target_idx = dist_order.index(target)
    fallback_dist = None
    fallback_seconds = None
    for d in dist_order[target_idx + 1:]:  # Kortare distanser
        if best_6w[d]['seconds'] is not None:
            fallback_dist = d
            fallback_seconds = best_6w[d]['seconds']
            break

    if fallback_dist is None:
        estimated_times.append('N/A')
        continue

    # Steg 3: Avrunda neråt i tabellen för fallback-distansen
    best_row = None
    for row in pace_table_seconds:
        if row[fallback_dist] <= fallback_seconds:
            best_row = row
        else:
            break

    if best_row is None:
        estimated_times.append('N/A')
        continue

    # Steg 4: Hämta target-distansens tid från den raden
    estimated_times.append(seconds_to_time(best_row[target]))




# Bygg tabelldata
pr_times = [pr_results[d]['time'] for d in table_distances]
pr_dates = [str(pr_results[d]['date']) for d in table_distances]
recent_times = [best_6w[d]['time'] for d in table_distances]
colors = [get_color(pr_results[d]['seconds'], best_6w[d]['seconds']) for d in table_distances]


fig_pr = go.Figure(data=[go.Table(
    header=dict(
        values=['Distans', 'PR', 'Datum PR', 'Best Effort Senaste 6v', 'Est. Race Time'],
        fill_color='darkgray',
        font=dict(color='white'),
        align='left'
    ),
    cells=dict(
        values=[table_distances, pr_times, pr_dates, recent_times, estimated_times],
        fill_color=[['white'] * 4, ['white'] * 4, ['white'] * 4, colors, ['white'] * 4],
        align='left'
    )
)])


fig_pr.update_layout(title='PR & Nuvarande Form')
fig_pr.show()


**`KPI 3: RUNNING HEATMAP`**


In [17]:
# @title

# Avkoda alla polylines till koordinater
def decode_polyline(poly):
    if poly:
        return polyline.decode(poly)
    return []

df['coordinates'] = df['summary_polyline'].apply(decode_polyline)


# Platta ut alla koordinater till en lista
all_coords = []
for coords in df['coordinates']:
    for lat, lon in coords:
        all_coords.append({'lat': lat, 'lon': lon})

df_coords = pd.DataFrame(all_coords)

fig_map = px.density_mapbox(
    df_coords,
    lat='lat',
    lon='lon',
    radius=3,
    center=dict(lat=59.31, lon=18.07),
    zoom=11,
    mapbox_style='carto-positron',
    color_continuous_scale='RdYlGn'
)

fig_map.update_layout(title='Löparheatmap')
fig_map.show()



**`KPI 4: PACE PER ZONE`**






Hämta detaljerad pace-data för relevanta pass

In [18]:
# @title
# Definiera jämförbara perioder
periods = {
    'Nu': (today - pd.Timedelta(weeks=4), today),
    '10v sedan': (today - pd.Timedelta(weeks=14), today - pd.Timedelta(weeks=10)),
    '26v sedan': (today - pd.Timedelta(weeks=30), today - pd.Timedelta(weeks=26)),
    '52v sedan': (today - pd.Timedelta(weeks=56), today - pd.Timedelta(weeks=52)),
}

# Samla ihop alla aktivitets-IDs för alla perioder
period_activity_ids = {}
for name, (start, end) in periods.items():
    ids = df[(df['start_date_local'] >= start) & (df['start_date_local'] < end)]['id'].tolist()
    period_activity_ids[name] = ids

all_ids = list(set([id for ids in period_activity_ids.values() for id in ids]))

# Hämta streams för alla aktiviteter
def get_streams(activity_id, access_token):
    response = requests.get(
        f"https://www.strava.com/api/v3/activities/{activity_id}/streams",
        headers={"Authorization": f"Bearer {access_token}"},
        params={"keys": "heartrate,velocity_smooth", "key_by_type": True},
        timeout=30
    )
    response.raise_for_status()
    return response.json()

all_streams = {}

for activity_id in all_ids:
    try:
        streams = get_streams(activity_id, access_token)
        all_streams[activity_id] = streams
    except requests.exceptions.HTTPError as e:
        if e.response.status_code == 429:
            print("Rate limit nådd, väntar 60 sekunder...")
            time.sleep(60)
            streams = get_streams(activity_id, access_token)
            all_streams[activity_id] = streams
        else:
            print(f"Fel för aktivitet {activity_id}: {e}")
    time.sleep(1)

Fel för aktivitet 18108917651: 404 Client Error: Not Found for url: https://www.strava.com/api/v3/activities/18108917651/streams?keys=heartrate%2Cvelocity_smooth&key_by_type=True


In [19]:
# @title
def get_zone(hr):
    if hr < 0.6 * HR_Max:
        return None
    elif hr < 0.7 * HR_Max:
        return 'Zon 1'
    elif hr < 0.8 * HR_Max:
        return 'Zon 2'
    elif hr < 0.9 * HR_Max:
        return 'Zon 3'
    elif hr < HR_Max:
        return 'Zon 4'
    else:
        return 'Zon 5'


# Bygg en dataframe med alla samples
all_samples = []

# För alla pass som inte saknar HR eller pace, spara HR, pace, och zon
for activity_id, streams in all_streams.items():
    if 'heartrate' not in streams or 'velocity_smooth' not in streams:
        continue

    hr_data = streams['heartrate']['data']
    vel_data = streams['velocity_smooth']['data']

    for hr, vel in zip(hr_data, vel_data):
        if vel > 0:  # Filtrera bort stillastående
            pace_sec_per_km = 1000 / vel  # Omvandla m/s till sek/km
            zone = get_zone(hr)
            if zone:
                all_samples.append({
                    'activity_id': activity_id,
                    'hr': hr,
                    'pace_sec_per_km': pace_sec_per_km,
                    'zone': zone
                })

df_samples = pd.DataFrame(all_samples)


# Koppla samples till perioder
def get_period(activity_id):
    activity_id = int(activity_id)
    for name, ids in period_activity_ids.items():
        if activity_id in ids:
            return name
    return None

df_samples['period'] = df_samples['activity_id'].apply(get_period)


results = []
for period in periods.keys():
    period_samples = df_samples[df_samples['period'] == period]
    for zone in ['Zon 1', 'Zon 2', 'Zon 3', 'Zon 4', 'Zon 5']:
        zone_samples = period_samples[period_samples['zone'] == zone]
        if len(zone_samples) > 0:
            avg_pace = zone_samples['pace_sec_per_km'].mean()
            time_spent = len(zone_samples)  # sekunder
            results.append({
                'period': period,
                'zone': zone,
                'avg_pace': seconds_to_pace(avg_pace),
                'avg_pace_seconds': avg_pace,
                'time_spent_seconds': time_spent,
                'time_spent': seconds_to_pace(time_spent)
            })

df_zone_results = pd.DataFrame(results)

Tabell över zon-paces atm

In [20]:
# @title
# Tabell 1: Nuvarande period - pace och tid per zon
zones = ['Zon 1', 'Zon 2', 'Zon 3', 'Zon 4', 'Zon 5']
nu_data = df_zone_results[df_zone_results['period'] == 'Nu']

tabell1 = []
for zone in zones:
    row = nu_data[nu_data['zone'] == zone]
    if len(row) > 0:
        tabell1.append({'Zon': zone, 'Avg Pace': row['avg_pace'].values[0], 'Tid': row['time_spent'].values[0]})
    else:
        tabell1.append({'Zon': zone, 'Avg Pace': 'N/A', 'Tid': 'N/A'})

df_tabell1 = pd.DataFrame(tabell1)


fig1 = go.Figure(data=[go.Table(
    header=dict(
        values=['Zon', 'Avg Pace', 'Tid'],
        fill_color='darkgray',
        font=dict(color='white'),
        align='left'
    ),
    cells=dict(
        values=[df_tabell1['Zon'], df_tabell1['Avg Pace'], df_tabell1['Tid']],
        fill_color='white',
        align='left'
    )
)])

fig1.update_layout(title='Avg Pace per Pulszon - Nuvarande period')
fig1.show()

Tabell över zon-paces historiskt

In [21]:
# @title
# Tabell 2: Jämförelse över tid
periods_order = ['52v sedan', '26v sedan', '10v sedan', 'Nu']
zones = ['Zon 1', 'Zon 2', 'Zon 3', 'Zon 4', 'Zon 5']

tabell2 = []
for zone in zones:
    row = {'Zon': zone}
    for period in periods_order:
        period_data = df_zone_results[(df_zone_results['period'] == period) & (df_zone_results['zone'] == zone)]
        if len(period_data) > 0:
            row[f'Pace {period}'] = period_data['avg_pace'].values[0]
        else:
            row[f'Pace {period}'] = 'N/A'
    nu_data = df_zone_results[(df_zone_results['period'] == 'Nu') & (df_zone_results['zone'] == zone)]
    row['Tid (Nu)'] = nu_data['time_spent'].values[0] if len(nu_data) > 0 else 'N/A'
    tabell2.append(row)

df_tabell2 = pd.DataFrame(tabell2)


# Färger per rad
pace_cols = ['Pace 52v sedan', 'Pace 26v sedan', 'Pace 10v sedan', 'Pace Nu']
col_colors = [['white'] * len(zones)]

row_colors = []
for _, row in df_tabell2.iterrows():
    paces = [row[col] for col in pace_cols]
    row_colors.append(get_row_colors(paces))

for col_idx in range(len(pace_cols)):
    col_colors.append([row_colors[row_idx][col_idx] for row_idx in range(len(zones))])
col_colors.append(['white'] * len(zones))

fig2 = go.Figure(data=[go.Table(
    header=dict(
        values=['Zon', 'Pace 52v sedan', 'Pace 26v sedan', 'Pace 10v sedan', 'Pace Nu', 'Tid (Nu)'],
        fill_color='darkgray',
        font=dict(color='white'),
        align='left'
    ),
    cells=dict(
        values=[
            df_tabell2['Zon'],
            df_tabell2['Pace 52v sedan'],
            df_tabell2['Pace 26v sedan'],
            df_tabell2['Pace 10v sedan'],
            df_tabell2['Pace Nu'],
            df_tabell2['Tid (Nu)']
        ],
        fill_color=col_colors,
        align='left'
    )
)])

fig2.update_layout(title='Avg Pace per Pulszon - Historisk jämförelse')
fig2.show()


Linjegraf - Pace per Zone



In [22]:
# @title
fig2_line = go.Figure()

for zone in zones:
    zone_data = df_zone_results[df_zone_results['zone'] == zone]
    # Sortera perioder i rätt ordning
    zone_data = zone_data.set_index('period').reindex(periods_order).reset_index()

    fig2_line.add_trace(go.Scatter(
        x=zone_data['period'],
        y=zone_data['avg_pace_seconds'],
        mode='lines+markers',
        name=zone,
        connectgaps=True
    ))

fig2_line.update_layout(
    title='Avg Pace per Pulszon - Historisk jämförelse',
    xaxis_title='Period',
    yaxis_title='Pace (sek/km)',
    yaxis=dict(
        tickmode='array',
        tickvals=list(range(300, 700, 30)),
        ticktext=[seconds_to_pace(s) for s in range(300, 700, 30)]
    )
)

fig2_line.show()

Cirkeldiagram över tid per pulszon

In [23]:
# @title
fig3 = go.Figure(data=[go.Pie(
    labels=df_tabell1['Zon'],
    values=[row['time_spent_seconds'] if row['time_spent_seconds'] != 'N/A' else 0
            for _, row in df_zone_results[df_zone_results['period'] == 'Nu'].iterrows()],
    hole=0.3
)])

fig3.update_layout(title='Tid per Pulszon - Nuvarande period')
fig3.show()

**`KPI 5: PACE PER DISTANCE`**


In [24]:
# @title
# Kolla pace och distans per pass i de fyra perioderna
def get_pace_from_activity(row):
    if row['average_speed'] > 0:
        return 1000 / row['average_speed']
    return None

df['pace_sec_per_km'] = df.apply(get_pace_from_activity, axis=1)

# Distansgrupper
def get_distance_group(km):
    if km < 5:
        return '< 5 km'
    elif km < 10:
        return '5-10 km'
    elif km < 15:
        return '10-15 km'
    else:
        return '15+ km'

df['distance_group'] = df['distance_km'].apply(get_distance_group)


# Koppla aktiviteter till perioder
def get_period_from_date(date):
    for name, (start, end) in periods.items():
        if start <= date < end:
            return name
    return None

df['period'] = df['start_date_local'].apply(get_period_from_date)



distance_groups = ['< 5 km', '5-10 km', '10-15 km', '15+ km']
df_period = df[df['period'].notna()]

results_dist = []
for period in periods.keys():
    period_df = df_period[df_period['period'] == period]
    for group in distance_groups:
        group_df = period_df[period_df['distance_group'] == group]
        if len(group_df) > 0:
            avg_pace = group_df['pace_sec_per_km'].mean()
            results_dist.append({
                'period': period,
                'distance_group': group,
                'avg_pace': seconds_to_pace(avg_pace),
                'avg_pace_seconds': avg_pace,
                'count': len(group_df)
            })
        else:
            results_dist.append({
                'period': period,
                'distance_group': group,
                'avg_pace': 'N/A',
                'avg_pace_seconds': None,
                'count': 0
            })

df_dist_results = pd.DataFrame(results_dist)


Tabell över paces atm

In [25]:
# @title
# Tabell 1: Nuvarande period
dist_nu = []
for group in distance_groups:
    row = df_dist_results[(df_dist_results['period'] == 'Nu') & (df_dist_results['distance_group'] == group)]
    if len(row) > 0 and row['avg_pace'].values[0] != 'N/A':
        dist_nu.append({'Distans': group, 'Avg Pace': row['avg_pace'].values[0], 'Antal pass': int(row['count'].values[0])})
    else:
        dist_nu.append({'Distans': group, 'Avg Pace': 'N/A', 'Antal pass': 0})

df_dist_tabell1 = pd.DataFrame(dist_nu)

fig_dist1 = go.Figure(data=[go.Table(
    header=dict(
        values=['Distans', 'Avg Pace', 'Antal pass'],
        fill_color='darkgray',
        font=dict(color='white'),
        align='left'
    ),
    cells=dict(
        values=[df_dist_tabell1['Distans'], df_dist_tabell1['Avg Pace'], df_dist_tabell1['Antal pass']],
        fill_color='white',
        align='left'
    )
)])

fig_dist1.update_layout(title='Avg Pace per Distansgrupp - Nuvarande period')
fig_dist1.show()

Tabell över paces historiskt

In [26]:
# @title
# Bygg jämförelsetabell
dist_tabell2 = []
for group in distance_groups:
    row = {'Distans': group}
    for period in periods_order:
        data = df_dist_results[(df_dist_results['period'] == period) & (df_dist_results['distance_group'] == group)]
        row[f'Pace {period}'] = data['avg_pace'].values[0] if len(data) > 0 else 'N/A'
    nu_data = df_dist_results[(df_dist_results['period'] == 'Nu') & (df_dist_results['distance_group'] == group)]
    row['Antal pass (Nu)'] = int(nu_data['count'].values[0]) if len(nu_data) > 0 else 0
    dist_tabell2.append(row)

df_dist_tabell2 = pd.DataFrame(dist_tabell2)


pace_cols = ['Pace 52v sedan', 'Pace 26v sedan', 'Pace 10v sedan', 'Pace Nu']
col_colors_dist = [['white'] * len(distance_groups)]

row_colors = []
for _, row in df_dist_tabell2.iterrows():
    paces = [row[col] for col in pace_cols]
    row_colors.append(get_row_colors(paces))

for col_idx in range(len(pace_cols)):
    col_colors_dist.append([row_colors[row_idx][col_idx] for row_idx in range(len(distance_groups))])
col_colors_dist.append(['white'] * len(distance_groups))

fig_dist2 = go.Figure(data=[go.Table(
    header=dict(
        values=['Distans', 'Pace 52v sedan', 'Pace 26v sedan', 'Pace 10v sedan', 'Pace Nu', 'Antal pass (Nu)'],
        fill_color='darkgray',
        font=dict(color='white'),
        align='left'
    ),
    cells=dict(
        values=[
            df_dist_tabell2['Distans'],
            df_dist_tabell2['Pace 52v sedan'],
            df_dist_tabell2['Pace 26v sedan'],
            df_dist_tabell2['Pace 10v sedan'],
            df_dist_tabell2['Pace Nu'],
            df_dist_tabell2['Antal pass (Nu)']
        ],
        fill_color=col_colors_dist,
        align='left'
    )
)])

fig_dist2.update_layout(title='Avg Pace per Distansgrupp - Historisk jämförelse')
fig_dist2.show()

Linjegraf - Pace per Distance


In [27]:
# @title
fig_dist2_line = go.Figure()

for group in distance_groups:
    group_data = df_dist_results[df_dist_results['distance_group'] == group]
    group_data = group_data.set_index('period').reindex(periods_order).reset_index()

    fig_dist2_line.add_trace(go.Scatter(
        x=group_data['period'],
        y=group_data['avg_pace_seconds'],
        mode='lines+markers',
        name=group,
        connectgaps=True
    ))

fig_dist2_line.update_layout(
    title='Avg Pace per Distansgrupp - Historisk jämförelse',
    xaxis_title='Period',
    yaxis_title='Pace (sek/km)',
    yaxis=dict(
        tickmode='array',
        tickvals=list(range(300, 700, 30)),
        ticktext=[seconds_to_pace(s) for s in range(300, 700, 30)]
    )
)

fig_dist2_line.show()

**Överblick kontra mål**

In [28]:
def get_gauge_color(value, goal):
    if goal == 0:
        return 'lightgray'
    ratio = min(value / goal, 1.0)
    if ratio < 0.5:
        r = 255
        g = int(200 * ratio * 2)
    else:
        r = int(255 * (1 - (ratio - 0.5) * 2))
        g = 200
    return f'rgb({r},{g},0)'

fig_header = make_subplots(
    rows=1, cols=2,
    specs=[[{'type': 'indicator'}, {'type': 'indicator'}]]
)

fig_header.add_trace(go.Indicator(
    mode='gauge+number',
    value=km_this_week,
    title={'text': 'Km denna vecka'},
    number={'suffix': ' km', 'valueformat': '.1f'},
    gauge={
        'axis': {'range': [0, weekly_mileage_goal]},
        'bar': {'color': get_gauge_color(km_this_week, weekly_mileage_goal)},
        'threshold': {
            'line': {'color': 'green', 'width': 4},
            'thickness': 0.75,
            'value': weekly_mileage_goal
        }
    }
), row=1, col=1)

fig_header.add_trace(go.Indicator(
    mode='gauge+number',
    value=runs_this_week,
    title={'text': 'Pass denna vecka'},
    gauge={
        'axis': {'range': [0, weekly_runs_goal]},
        'bar': {'color': get_gauge_color(runs_this_week, weekly_runs_goal)},
        'threshold': {
            'line': {'color': 'green', 'width': 4},
            'thickness': 0.75,
            'value': weekly_runs_goal
        }
    }
), row=1, col=2)

fig_header.update_layout(height=300)
fig_header.show()

**EXPORTERA TILL HTML**

In [29]:
import plotly.io as pio

def fig_to_html(fig):
    return fig.to_html(full_html=False, include_plotlyjs=False)

html = f"""
<!DOCTYPE html>
<html lang="sv">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Running Dashboard</title>
    <script src="https://cdn.plot.ly/plotly-latest.min.js"></script>
    <style>
        body {{
            font-family: Arial, sans-serif;
            max-width: 1400px;
            margin: 0 auto;
            padding: 20px;
            background-color: #f5f5f5;
        }}
        h1 {{
            text-align: center;
            color: #333;
        }}
        h2 {{
            color: #555;
            border-bottom: 2px solid #ddd;
            padding-bottom: 8px;
        }}
        .section {{
            background: white;
            border-radius: 8px;
            padding: 20px;
            margin-bottom: 30px;
            box-shadow: 0 2px 4px rgba(0,0,0,0.1);
        }}
    </style>
</head>
<body>
    <h1>🏃 Running Dashboard</h1>

    <div class="section">
        <h2>Veckoöversikt</h2>
        {fig_to_html(fig_header)}
    </div>

    <div class="section">
        <h2>Km per vecka</h2>
        {fig_to_html(fig_km)}
    </div>

    <div class="section">
        <h2>Pass per vecka</h2>
        {fig_to_html(fig_runs)}
    </div>

    <div class="section">
        <h2>PR & Nuvarande Form</h2>
        {fig_to_html(fig_pr)}
    </div>

    <div class="section">
        <h2>Avg Pace per Pulszon</h2>
        {fig_to_html(fig2)}
        {fig_to_html(fig2_line)}
        {fig_to_html(fig3)}
    </div>

    <div class="section">
        <h2>Avg Pace per Distansgrupp</h2>
        {fig_to_html(fig_dist2)}
        {fig_to_html(fig_dist2_line)}
    </div>

    <div class="section">
        <h2>Löparheatmap</h2>
        {fig_to_html(fig_map)}
    </div>

</body>
</html>
"""

with open('running_dashboard.html', 'w', encoding='utf-8') as f:
    f.write(html)

print("Dashboard sparad!")

Dashboard sparad!
